# PyPSA-Iberia 2025 Dispatch Notebook: Model Results vs Actual Data

This notebook is for the **dispatch-style 2025 Iberia run** and is designed around two jobs:

1. explore what the PyPSA-Eur result says about dispatch, congestion, and especially prices
2. line those model results up with **actual market and system data** where possible

The emphasis is on **pricing results** because they are a compact summary of scarcity, fuel costs, renewable output, and network limits.

The notebook is also written to be careful about stale outputs. If the saved network does **not** actually contain 2025 snapshots, it will warn explicitly instead of pretending it is a true 2025-weather simulation.


## Planned Comparison Data

Recommended external sources for comparison:

- **OMIE / REN**: Iberian day-ahead prices for Spain and Portugal
- **ESIOS (REE)**: Spain demand, Spain generation by technology, and some market indicators
- **ENTSO-E Transparency Platform**: alternative day-ahead price source for bidding zones, useful if you already have an API token
- **SARAH-3 / ERA5 context**: the model weather inputs are based on satellite/reanalysis products, so actual comparisons should focus on the resulting **market outcomes** and **realized generation**, not only on abstract weather series

Because some official APIs require credentials, the notebook includes download helpers and fallbacks rather than assuming access is already configured.


In [ ]:
from pathlib import Path
import io
import os
import warnings

import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pypsa
import requests
import seaborn as sns
import yaml
from IPython.display import Markdown, display

plt.style.use("bmh")
sns.set_theme(style="whitegrid")
pd.options.display.float_format = "{:,.2f}".format


def find_repo_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "results").exists() and (candidate / "config").exists():
            return candidate
    raise FileNotFoundError("Could not locate repository root.")


ROOT = find_repo_root()
RUN_NAME = "iberia-dispatch-2025"
NETWORK_CANDIDATES = [
    ROOT / f"results/{RUN_NAME}/networks/base_s_9_elec_.nc",
    ROOT / f"results/{RUN_NAME}/networks/base_s_9_elec_Ept80.nc",
]
CONFIG_CANDIDATES = [
    ROOT / "config/iberia/config.dispatch2025-weather.yaml",
    ROOT / "config/iberia/config.dispatch2025.yaml",
]


def first_existing(paths: list[Path]) -> Path | None:
    for path in paths:
        if path.exists():
            return path
    return None


network_path = first_existing(NETWORK_CANDIDATES)
config_path = first_existing(CONFIG_CANDIDATES)

if network_path is None:
    raise FileNotFoundError("No dispatch-2025 network found. Update NETWORK_CANDIDATES.")

n = pypsa.Network(network_path)
cfg = yaml.safe_load(config_path.read_text()) if config_path else {}

display(Markdown(f"**Network file**: `{network_path}`  \\n**Config file**: `{config_path}`"))
print("Snapshots in saved network:", n.snapshots[0], "to", n.snapshots[-1], f"({len(n.snapshots)} hours)")
if cfg:
    print("Configured snapshot window:", cfg['snapshots']['start'], "to", cfg['snapshots']['end'])


In [ ]:
def configured_year(config: dict) -> int | None:
    try:
        return pd.Timestamp(config["snapshots"]["start"]).year
    except Exception:
        return None


def snapshot_year(network: pypsa.Network) -> int:
    return pd.Timestamp(network.snapshots[0]).year


configured = configured_year(cfg)
actual = snapshot_year(n)

if configured is not None and actual != configured:
    warnings.warn(
        f"This saved network spans {n.snapshots[0]} to {n.snapshots[-1]}, not the configured {configured} window. "
        "Treat this as a stale or earlier-weather result until the workflow is rerun.",
        stacklevel=2,
    )
    display(
        Markdown(
            f"**Warning**: the network file uses **{actual}** snapshots (`{n.snapshots[0]}` to `{n.snapshots[-1]}`), "
            f"while the weather config requests **{configured}** (`{cfg['snapshots']['start']}` to `{cfg['snapshots']['end']}`)."
        )
    )
else:
    display(Markdown(f"**Snapshot year check passed**: network and config both point to **{actual}**."))


## 1. Model Overview

Start by checking what kind of system this dispatch result actually contains: technologies, spatial resolution, and the overall price level.


In [ ]:
def mean_ac_prices(network: pypsa.Network) -> pd.Series:
    buses = network.buses.index[network.buses.carrier == "AC"]
    return network.buses_t.marginal_price[buses].mean().sort_values(ascending=False)


overview = pd.Series(
    {
        "buses": len(n.buses),
        "lines": len(n.lines),
        "generators": len(n.generators),
        "storage_units": len(n.storage_units),
        "snapshots": len(n.snapshots),
        "annual_load_twh": n.loads_t.p_set.sum().sum() / 1e6,
        "mean_ac_price_eur_per_mwh": mean_ac_prices(n).mean(),
    }
).rename("value")
display(overview.to_frame())

print("Generator carriers")
display(n.generators.carrier.value_counts().rename("count").to_frame())

print("Storage carriers")
display(n.storage_units.carrier.value_counts().rename("count").to_frame())


## 2. Price Exploration

This is the main section. Good ways to explore model prices include:

- system-wide price duration curves
- monthly and hourly seasonality
- nodal price differences and price spreads
- price heatmaps over the year
- price vs load, VRE output, and net load
- capture prices / market values by technology
- identification of price spike hours and the system conditions behind them


In [ ]:
prices = n.buses_t.marginal_price[n.buses.index[n.buses.carrier == "AC"]]
system_price = prices.mean(axis=1)
price_duration = system_price.sort_values(ascending=False).reset_index(drop=True)
monthly_price = system_price.groupby(system_price.index.month).mean()
hourly_price_shape = system_price.groupby(system_price.index.hour).mean()
weekday_price_shape = system_price.groupby(system_price.index.day_name()).mean().reindex(
    ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
)

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes[0, 0].plot(price_duration.values, color="navy")
axes[0, 0].set_title("System price duration curve")
axes[0, 0].set_xlabel("hour rank")
axes[0, 0].set_ylabel("EUR/MWh")

axes[0, 1].plot(monthly_price.index, monthly_price.values, marker="o", color="darkred")
axes[0, 1].set_title("Average system price by month")
axes[0, 1].set_xlabel("month")
axes[0, 1].set_ylabel("EUR/MWh")

axes[1, 0].plot(hourly_price_shape.index, hourly_price_shape.values, marker="o", color="darkorange")
axes[1, 0].set_title("Average system price by hour of day")
axes[1, 0].set_xlabel("hour")
axes[1, 0].set_ylabel("EUR/MWh")

axes[1, 1].bar(weekday_price_shape.index, weekday_price_shape.values, color="steelblue")
axes[1, 1].set_title("Average system price by weekday")
axes[1, 1].tick_params(axis="x", rotation=45)
axes[1, 1].set_ylabel("EUR/MWh")

plt.tight_layout()
plt.show()

display(system_price.describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).to_frame("system_price_eur_per_mwh"))


In [ ]:
weekday_hour = (
    system_price.to_frame("price")
    .assign(weekday=system_price.index.day_name(), hour=system_price.index.hour)
    .pivot_table(index="weekday", columns="hour", values="price", aggfunc="mean")
    .reindex(["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"])
)

fig, ax = plt.subplots(figsize=(11, 4.8))
sns.heatmap(weekday_hour, cmap="Spectral_r", annot=False, ax=ax)
ax.set_title("Average price by weekday and hour")
ax.set_xlabel("hour of day")
ax.set_ylabel("")
plt.show()


In [ ]:
def unstack_day_hour(s: pd.Series) -> pd.DataFrame:
    grouped = s.groupby(s.index.hour).agg(list)
    index = [f"{h:02d}:00" for h in grouped.index]
    columns = pd.date_range(s.index[0], s.index[-1], freq="D")
    if len(columns) != len(grouped.iloc[0]):
        columns = columns[: len(grouped.iloc[0])]
    return pd.DataFrame(grouped.to_list(), index=index, columns=columns)


fig, ax = plt.subplots(figsize=(12, 4.8))
sns.heatmap(unstack_day_hour(system_price), cmap="Spectral_r", ax=ax)
ax.set_title("System price heatmap across the year")
ax.set_xlabel("day of year")
ax.set_ylabel("hour")
plt.show()


In [ ]:
bus_price_table = pd.DataFrame(
    {
        "mean_price": prices.mean(),
        "std_price": prices.std(),
        "min_price": prices.min(),
        "max_price": prices.max(),
    }
).sort_values("mean_price", ascending=False)
display(bus_price_table)

spread_to_mean = prices.sub(system_price, axis=0)
spread_summary = pd.DataFrame(
    {
        "avg_spread_to_system": spread_to_mean.mean(),
        "std_spread_to_system": spread_to_mean.std(),
        "p95_abs_spread": spread_to_mean.abs().quantile(0.95),
    }
).sort_values("p95_abs_spread", ascending=False)
display(spread_summary)


In [ ]:
load = n.loads_t.p_set.sum(axis=1) / 1e3
vre_carriers = ["solar", "onwind", "offwind-ac", "offwind-float", "ror"]
vre = n.generators_t.p[n.generators.index[n.generators.carrier.isin(vre_carriers)]].sum(axis=1) / 1e3
net_load = load - vre

corr_table = pd.Series(
    {
        "corr(price, load)": system_price.corr(load),
        "corr(price, VRE)": system_price.corr(vre),
        "corr(price, net_load)": system_price.corr(net_load),
    }
).rename("correlation")
display(corr_table.to_frame())

fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))
axes[0].scatter(load, system_price, s=6, alpha=0.35, color="darkred")
axes[0].set_title("Price vs load")
axes[0].set_xlabel("load [GW]")
axes[0].set_ylabel("price [EUR/MWh]")

axes[1].scatter(vre, system_price, s=6, alpha=0.35, color="seagreen")
axes[1].set_title("Price vs VRE output")
axes[1].set_xlabel("VRE [GW]")
axes[1].set_ylabel("price [EUR/MWh]")

axes[2].scatter(net_load, system_price, s=6, alpha=0.35, color="navy")
axes[2].set_title("Price vs net load")
axes[2].set_xlabel("net load [GW]")
axes[2].set_ylabel("price [EUR/MWh]")

plt.tight_layout()
plt.show()


In [ ]:
market_values = n.statistics.market_value(bus_carrier="AC", aggregate_across_components=True).dropna()
market_values = market_values[~market_values.index.isin(["AC", "-"])]
mv = market_values.sort_values(ascending=False)
display(mv.to_frame("market_value_eur_per_mwh"))

fig, ax = plt.subplots(figsize=(8, 4.8))
mv.sort_values().plot.barh(ax=ax, color="mediumpurple")
ax.set_title("Technology market values")
ax.set_xlabel("EUR/MWh")
plt.show()


In [ ]:
top_spikes = system_price.nlargest(24).rename("price_eur_per_mwh")
spike_context = pd.DataFrame(
    {
        "price_eur_per_mwh": top_spikes,
        "load_gw": load.reindex(top_spikes.index),
        "vre_gw": vre.reindex(top_spikes.index),
        "net_load_gw": net_load.reindex(top_spikes.index),
    }
).sort_index()
display(spike_context)


## 3. Dispatch and Balancing Context

Price interpretation is much easier if you also look at the system dispatch. For a dispatch-style run, the useful views are annual energy balance, a few representative weeks, and storage/hydro behavior.


In [ ]:
annual_energy = (n.statistics.energy_balance(groupby="carrier") / 1e6).sort_values(ascending=False)
display(annual_energy.to_frame("twh"))

gen_dispatch = n.generators_t.p.T.groupby(n.generators.carrier).sum().T / 1e3
store_dispatch = n.storage_units_t.p.clip(lower=0).T.groupby(n.storage_units.carrier).sum().T / 1e3
dispatch = pd.concat([gen_dispatch, store_dispatch], axis=1).groupby(level=0, axis=1).sum()
major = dispatch.mean().sort_values(ascending=False).head(6).index
dispatch_plot = dispatch.copy()
dispatch_plot["other"] = dispatch_plot.drop(columns=major, errors="ignore").sum(axis=1)
dispatch_plot = dispatch_plot[list(major) + ["other"]]

week_start = str(n.snapshots[0].date())
week_end = str((n.snapshots[0] + pd.Timedelta(days=6)).date())
window = slice(week_start, week_end)

fig, ax = plt.subplots(figsize=(12, 4.8))
dispatch_plot.loc[window].plot.area(ax=ax, linewidth=0)
ax.plot(load.loc[window].index, load.loc[window].values, color="black", linewidth=2, label="load")
ax.set_title(f"Dispatch stack: {week_start} to {week_end}")
ax.set_ylabel("GW")
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1.0))
plt.tight_layout()
plt.show()


## 4. Actual Data Hooks

The cells below are designed to download or ingest actual external data. Some sources need credentials.

Recommended workflow:

- use **ESIOS** if you have a token and want demand / generation / some market series for Spain
- use **OMIE or REN exports** for Iberian day-ahead price comparison
- use **ENTSO-E** if you already have a token and want bidding-zone day-ahead prices through that route instead


In [ ]:
ESIOS_BASE = "https://api.esios.ree.es"
ESIOS_TOKEN = os.getenv("ESIOS_API_TOKEN")


def esios_headers(token: str | None = None) -> dict:
    token = token or ESIOS_TOKEN
    if not token:
        raise RuntimeError("Set ESIOS_API_TOKEN in your environment before calling the ESIOS helpers.")
    return {
        "Accept": "application/json; application/vnd.esios-api-v1+json",
        "Content-Type": "application/json",
        "x-api-key": token,
    }


def esios_search_indicators(text: str, taxonomy_ids: list[int] | None = None, token: str | None = None) -> pd.DataFrame:
    params = {"text": text}
    if taxonomy_ids:
        params["taxonomy_ids[]"] = taxonomy_ids
    r = requests.get(f"{ESIOS_BASE}/indicators", headers=esios_headers(token), params=params, timeout=60)
    r.raise_for_status()
    data = r.json().get("indicators", [])
    return pd.DataFrame(data)


def esios_get_indicator(indicator_id: int, start: str, end: str, geo_ids: list[int] | None = None, token: str | None = None) -> pd.DataFrame:
    params = {
        "start_date": start,
        "end_date": end,
        "time_agg": "sum",
        "locale": "en",
    }
    if geo_ids:
        params["geo_ids[]"] = geo_ids
    r = requests.get(f"{ESIOS_BASE}/indicators/{indicator_id}", headers=esios_headers(token), params=params, timeout=60)
    r.raise_for_status()
    values = r.json().get("indicator", {}).get("values", [])
    df = pd.DataFrame(values)
    if not df.empty and "datetime" in df.columns:
        df["datetime"] = pd.to_datetime(df["datetime"])
    return df


display(Markdown("Use `esios_search_indicators('price')`, `esios_search_indicators('demand')`, `esios_search_indicators('wind')`, and `esios_search_indicators('solar')` to find the current official indicator IDs before downloading."))


In [ ]:
# Example ESIOS searches. Run these only if you have an API token.
# display(esios_search_indicators('day-ahead price').head(20))
# display(esios_search_indicators('demand').head(20))
# display(esios_search_indicators('wind').head(20))
# display(esios_search_indicators('solar').head(20))


In [ ]:
def load_actual_price_csv(path: str | Path, datetime_col: str = "datetime", price_col: str = "price_eur_per_mwh") -> pd.DataFrame:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    if path.suffix.lower() in {".xlsx", ".xls"}:
        df = pd.read_excel(path)
    else:
        try:
            df = pd.read_csv(path)
        except UnicodeDecodeError:
            df = pd.read_csv(path, encoding="latin1", sep=";")
    df.columns = [c.strip() for c in df.columns]
    if datetime_col in df.columns:
        df[datetime_col] = pd.to_datetime(df[datetime_col])
    return df


display(Markdown(
    "If you export OMIE or REN price files manually, point `load_actual_price_csv()` at the downloaded file and then map its columns to a common hourly timestamp / price format."
))


## 5. Model vs Actual Price Comparison Template

Once you have actual hourly day-ahead prices, the comparison should go beyond a single annual average. Useful diagnostics include:

- bias in average price
- correlation of hourly prices
- comparison of monthly means
- comparison of hourly price-shape by weekday
- comparison of price duration curves
- errors specifically during high-price and low-price hours


In [ ]:
# Replace this with your actual imported price file once available.
# actual_prices = load_actual_price_csv('data/actuals/omie_prices_2025.csv', datetime_col='datetime', price_col='price')
# actual_hourly = actual_prices.set_index('datetime')['price'].sort_index()

def compare_model_and_actual(model: pd.Series, actual: pd.Series, label_model: str = 'model', label_actual: str = 'actual'):
    df = pd.concat([model.rename(label_model), actual.rename(label_actual)], axis=1).dropna()
    if df.empty:
        raise ValueError('No overlapping timestamps between model and actual series.')
    df['error'] = df[label_model] - df[label_actual]
    stats = pd.Series(
        {
            'n_hours': len(df),
            'mean_model': df[label_model].mean(),
            'mean_actual': df[label_actual].mean(),
            'bias': df['error'].mean(),
            'mae': df['error'].abs().mean(),
            'rmse': np.sqrt((df['error'] ** 2).mean()),
            'corr': df[label_model].corr(df[label_actual]),
        }
    )
    return df, stats


# Example usage after loading actual data:
# cmp, cmp_stats = compare_model_and_actual(system_price, actual_hourly)
# display(cmp_stats.to_frame('value'))
# fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))
# cmp[[ 'model', 'actual' ]].sort_values('actual', ascending=False).reset_index(drop=True).plot(ax=axes[0])
# cmp.groupby(cmp.index.month)[['model', 'actual']].mean().plot(ax=axes[1], marker='o')
# axes[2].scatter(cmp['actual'], cmp['model'], s=6, alpha=0.35)
# plt.show()


## 6. Model vs Actual Demand / Generation Comparison Ideas

If you obtain actual demand and generation series from ESIOS, useful comparisons include:

- total demand profile vs model demand profile
- actual wind generation vs modeled onshore+offshore wind dispatch
- actual solar generation vs modeled solar dispatch
- actual hydro / pumped storage behavior vs modeled hydro / PHS schedules
- actual price vs model price conditional on demand quantiles or VRE quantiles

A good teaching point is that a mismatch in price may come from more than one source: demand assumptions, generator availability, fuel prices, network simplification, hydro constraints, and the fact that the model may not reproduce all market rules.


## 7. Notes on Weather Inputs

The weather-driven PyPSA-Eur setup does **not** use synthetic typical profiles here. It is intended to use time-resolved external weather products. In this configuration the cutout points to `europe-2025-sarah3-era5`, which means the renewable availability is meant to be informed by actual weather-year data products rather than the earlier 2013 tutorial-style weather year.

For comparison, however, the most robust external validation target is still usually **market and system outcomes**:

- actual day-ahead prices
- actual demand
- actual wind generation
- actual solar generation

That is usually more informative for students than trying to directly compare cutout weather variables cell by cell.


## Suggested Next Steps

1. Rerun the weather-based dispatch workflow so the saved network actually spans 2025 if that is the intended experiment.
2. Use the ESIOS helper cells to identify demand / solar / wind / price indicator IDs.
3. Export OMIE or REN hourly day-ahead price data and feed it into the comparison template.
4. Keep the model-vs-actual section focused on **where** and **when** the model departs from reality, not only on annual averages.
